### Imports

In [1]:
import torch
import torch.nn as nn
import copy
import random
import numpy as np
import torch.optim as optim
from tqdm.notebook import tqdm, trange
from torchvision import datasets, transforms
import time
import gc
from GPUtil import showUtilization as gpu_usage
from GPUtil import getGPUs
from torchvision.transforms import v2
import os, ctypes
from IPython.core.display import HTML
import os
import shutil
import json
from pathlib import Path

# For the correct functioning of tqdm
display(HTML("""
    <style>
        .jp-OutputArea-child:has(.jp-OutputArea-prompt:empty) {
              padding: 0 !important;
        }
    </style>
"""))

# To supress pesky warnings
import warnings
warnings.filterwarnings("ignore")

### Reproducibility

In [2]:
def set_seed(seed_value=42):
    np.random.seed(seed_value) # cpu vars
    random.seed(seed_value) # Python
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    os.environ['CUBLAS_WORKSPACE_CONFIG']=":4096:8"
    torch.manual_seed(seed_value) # cpu  vars
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.cuda.manual_seed_all(seed_value) # gpu vars
    torch.backends.cudnn.deterministic = True  #needed
    torch.backends.cudnn.benchmark = False

# 0. Useful functions

In this section we define all the functions we need, from creating and training the models to generating the datasets or cleaning unused memory.

### 0.1. Memory management

In [3]:
# Here we define a function to free all possible memory from the GPU to avoid memory issues when dealing with a lot of models
def free_gpu_cache(show_usage = False):
# This function is used to clear the GPU cache and avoid memory problems when dealing with large populations and big models
    if show_usage:
        print("Initial GPU Usage")
        gpu_usage()
    torch.cuda.ipc_collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()
    if show_usage:
        print("GPU Usage after emptying the cache")
        gpu_usage()

### 0.2. Model training functions

In [4]:
def mut_trainloop(base_model, mutations = 0, seed = 0, transforms = [], max_epochs = 256, lr = 1e-3, patience = 0, model_save_path = ""):
    try:
        accs = {"train" : [],
                "val" : [],
                "test" : []
               }
        times = []
        with tqdm(total = len(transforms), leave = False, desc='Data Augmentation scheme') as pbar0:
            for t_idx, t in enumerate(transforms):
                if type(base_model) == str:
                    model = torch.load(base_model+"data_aug_"+str(t_idx)+"/0.pt")
                else:
                    model = copy.deepcopy(base_model)

                if mutations:
                    set_seed(seed_value)
                    model.mutate(mut_amount = mutations, probs = {"add_b" : 0.4, "delete_b" : 0.4, "modify_b" : 0.2}, verbose = True)
                
                if t_idx == len(transforms)-1: # For the no Data Aug. with reinitialized weights case
                    model.apply(weight_reset)
                
                start = time.time()
                val_loader, train_loader = retrieve_datasets(t)
        
                model.to(model.device)
                optimizer = optim.AdamW(model.parameters(), lr)
                
                model.train()
                training_loss = []
                validation_loss = []
                overfit = 0
                best_loss = np.inf
                with tqdm(total = max_epochs , leave = False, desc='Training model') as pbar1:
                    for epoch in range(max_epochs ):
                        running_loss = 0.0
                        with tqdm(total=len(train_loader), leave = False, desc='Epoch progress') as pbar2:
                            for inputs, labels in train_loader:
                                time_start = time.time()
                                inputs, labels = inputs.to(model.device), labels.to(model.device)
                                optimizer.zero_grad()
                                outputs = model(inputs).squeeze()
                                loss = criterion(outputs, labels)
                                #print(loss)
                                loss.backward()
                                optimizer.step()
                                running_loss += loss.item()
    
                                if time.time() - time_start > model.max_iter_time:
                                    pbar0.update(np.inf)
                                    pbar1.update(np.inf)
                                    pbar2.update(np.inf)
                                    del model, optimizer
                                    free_gpu_cache()
                                    raise Exception("This model would take too much time to train. Trying another random model.")
                                pbar2.update(1)
                        training_loss.append(running_loss/len(train_loader))
                        
                        model.eval()
                        running_loss = 0.
                        with tqdm(total=len(val_loader), leave = False, desc='Calculating validation loss') as pbar3:
                            with torch.no_grad():
                                for inputs, labels in val_loader:
                                    inputs, labels = inputs.to(model.device), labels.to(model.device)
                                    outputs = model.forward(inputs)
                                    loss = criterion(outputs, labels)
                                    running_loss += loss.detach().cpu().numpy()
                                    pbar3.update(1)
                        validation_loss.append(running_loss/len(val_loader))
                
                        if patience > 0:
                            if validation_loss[-1] < best_loss:
                                best_loss = validation_loss[-1]
                                overfit = 0
                                torch.save(model.state_dict(), "training_model.pt")
                            else:
                                overfit += 1
                            if overfit >= patience:
                                pbar1.update(max_epochs )
                                break
                        pbar1.update(1)
                    
                if patience > 0:
                    model.load_state_dict(torch.load("training_model.pt"))
            
                accs["train"] = accs["train"] + [accmetric(model, train_loader)]
                accs["val"] = accs["val"] + [accmetric(model, val_loader)]
                accs["test"] = accs["test"] + [accmetric(model, test_loader)]
                
                torch.save(model, "data_aug_"+str(t_idx)+".pt")
                
                del model, optimizer, loss, inputs, labels, outputs
                times += [time.time()-start]
                
                pbar0.update()
                free_gpu_cache()

        print("Achieved accuracies:", accs)
        print("Training time:", times)
        valid = np.sum([accs["val"][i]>0.2 for i in range(len(transforms))]) == len(transforms)
        if valid:
            for t_idx in range(len(transforms)):
                shutil.copyfile("data_aug_"+str(t_idx)+".pt", model_save_path+"data_aug_"+str(t_idx)+"/"+str(len(os.listdir(model_save_path+"data_aug_"+str(t_idx))))+".pt")
            return accs, times
        else:
            return None, None
    
    except Exception as e:
        pbar0.update(np.inf)
        pbar1.update(np.inf)
        pbar2.update(np.inf)
        print(e)
        del model, optimizer
        free_gpu_cache()
        raise Exception("This model would not fit in the GPU. Trying another random model.")

def accmetric(model, dataloader):
    model.to(model.device)
    model.eval()
    
    accuracy = 0
    # Turn off gradients for validation, saves memory and computations
    with tqdm(total=len(dataloader), leave = False, desc='Calculating accuracy') as pbar:
        with torch.no_grad():
            for inputs, labels in dataloader:
                inputs, labels = inputs.to(model.device), labels.to(model.device)
                logprobs = model.forward(inputs)
                top_p, top_class = logprobs.topk(1, dim=1)
                equals = (top_class == labels.view(inputs.shape[0], 1))
                accuracy += torch.mean(equals.type(torch.FloatTensor))
                pbar.update(1)
    model.to("cpu")
    free_gpu_cache()
    return accuracy.detach().numpy()/len(dataloader)


### 0.3. Model generation

In [5]:
class conv_block(nn.Module):
    def __init__(self, in_c, out_c, kernel_size = 3):
        self.kernel_size = kernel_size
        padding = int((kernel_size + 1)/2)
        self.padding = padding
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU()
        
    def forward(self, inputs):
        x = self.conv(inputs)
        x = self.bn(x)
        x = self.relu(x)
        return x  

class block(nn.Module):
    def __init__(self, in_c, out_c, resizing_type = None, from_block_channels = 0, kernel_size = 3):
        super().__init__()
        self.resizing_type = resizing_type
        self.kernel_size = kernel_size
        self.in_c = in_c
        self.out_c = out_c
        self.from_block_channels = from_block_channels
        
        if resizing_type == "D":
            self.resizing = nn.MaxPool2d((2, 2))
        elif resizing_type == "U":
            self.resizing = nn.ConvTranspose2d(in_c, in_c, kernel_size=2, stride=2, padding=0)
        else:
            self.resizing = nn.Identity()
        
        self.conv = conv_block(from_block_channels+in_c, out_c, kernel_size = kernel_size)
        
    def forward(self, inputs, skip):
        x = self.resizing(inputs)
        
        for extra_input in skip:
            x = torch.cat([x, nn.functional.interpolate(extra_input, size = [x.shape[2], x.shape[3]])], axis=1)
        x = self.conv(x)
        return x

class optimized_network(nn.Module):
    """
    A Convolutional Neural Network generated randomly.
    """
    def __init__(self, dataloader, n_blocks = 1, task = "infere", out_size = None, logmin_c = 5, logmax_c = 8, max_k = 9, complex_classifier = False, max_iter_time = 1, verbose = False):
        super().__init__()
        
        if hasattr(n_blocks, '__iter__'):
            if len(n_blocks) not in [1,2]:
                raise Exception("Invalid range for model initial layers.")
        elif n_blocks == int(n_blocks):
            n_blocks = [int(n_blocks), int(n_blocks)]
        else:
            raise Exception("Model initial layers needs to be either a scalar or a scalar range.")

        self.dataloader = dataloader
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.logmin_c = logmin_c
        self.logmax_c = logmax_c
        self.max_k = max_k
        self.complex_classifier = complex_classifier
        self.max_iter_time = max_iter_time
        
        batch = next(iter(dataloader))
        self.in_size = batch[0].shape
        self.label_size = batch[1].shape
        
        targets = dataloader.dataset.targets
        
        if task not in ["image_to_image", "image_to_mask", "object_detection", "regression", "classification", "infere"]:
            print("Unknown task. Infering task from dataset.")
            task = "infere"
        if task == "infere":
            if batch[0].shape[-2:] == batch[1].shape[-2:]:
                if batch[1].is_floating_point():
                    self.task = "image_to_image"
                else:
                    if batch[1].shape[-3] == 1:
                        self.task = "image_to_mask"
                    else:
                        self.task = "object_detection"
            else:
                if batch[1].is_floating_point():
                    self.task = "regression"
                else:
                    self.task = "classification"
        else:
            self.task = task

        if out_size:
            self.out_size = out_size
        else:
            if self.task == "classification" or self.task == "object_detection":
                self.out_size = max(targets) + 1
            else:
                self.out_size = batch[1].shape[1:]
        
        invalid = self.initialize_network(n_blocks = random.randint(n_blocks[0], n_blocks[1]), verbose = verbose)
        while invalid:
            del self.features, self.fc
            free_gpu_cache()
            invalid = self.initialize_network(n_blocks = random.randint(n_blocks[0], n_blocks[1]), verbose = verbose)

    def calculate_possible_connections(self):
        updown = [0]
        for l in self.features:
            if type(l.resizing) == nn.Identity:
                r = 0
            elif type(l.resizing) == nn.ConvTranspose2d:
                r = 1
            elif type(l.resizing) == nn.MaxPool2d:
                r = -1
            updown = updown + [updown[-1] + r]
        updown = updown[1:]
    
        possible_connections = []
        for idxin, lin in enumerate(updown):
            for idxout, lout in enumerate(updown[idxin+2:]):
                if lin == lout:
                    possible_connections += [(idxin, idxin+2+idxout)]
        
        return possible_connections
    
    def correct_blocks(self):
        for l, blk in enumerate(list(self.features.children())): # We check each block's new input size, rebuilding it if necessary
            if l == 0:
                if not blk.in_c == self.in_size[1]:
                    self.features[l] = block(in_c = self.in_size[1], out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = 0, kernel_size = blk.kernel_size)
    
            else:
                from_blocks = [i for i, x in enumerate(self.connections) if l in x]
                from_block_channels = 0
                for from_block in from_blocks:
                    from_block_channels += self.features[from_block].out_c
                
                if not (blk.in_c == self.features[l-1].out_c and blk.from_block_channels == from_block_channels):
                    self.features[l] = block(in_c = self.features[l-1].out_c, out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = from_block_channels, kernel_size = blk.kernel_size)
        try:
            self.build_fc()
        except:
            raise Exception("This model would not fit in the GPU. Trying another random model.")
    
    def initialize_network(self, n_blocks, verbose):
        try:
            self.connections = [[]] * n_blocks
            self.features = nn.Sequential()
            self.genome = []
            for l in range(n_blocks):
                if l == 0:
                    in_c = self.in_size[1]
                else:
                    in_c = out_c
                
                if l == n_blocks-1 and self.task != "classification":
                    out_c = self.out_size
                else:
                    out_c = 2**random.randint(self.logmin_c, self.logmax_c)
                
                resizing_type = random.choice(["U", "D", ""])
                
                kernel_size = random.randint(0,int((self.max_k-1)/2))*2+1
                self.features = nn.Sequential(*list(self.features.children()), block(in_c = in_c, out_c = out_c, resizing_type = resizing_type, from_block_channels = 0, kernel_size = kernel_size))

            possible_connections = self.calculate_possible_connections()
            k = random.randint(0, len(possible_connections))
            initial_connections = random.sample(possible_connections, k=k)

            for c in initial_connections:
                self.connections[c[0]] = self.connections[c[0]] + [c[1]]
            
            for i in range(len(self.connections)):
                self.connections[i].sort() # We order the connections so that they are tidy

            self.correct_blocks()
            return False
        except Exception as e:
            if verbose:
                print(e)
            return True

    def build_fc(self):
        self.fc = nn.Sequential(nn.Identity())
        if self.task == "classification":
            feat_size = np.prod([*self.to(self.device).forward(torch.zeros(self.in_size, device = self.device)).shape[1:]])
            free_gpu_cache()
            if self.complex_classifier:
                self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size*4), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*4, self.out_size*2), nn.ReLU(), nn.Dropout(0.5), nn.Linear(self.out_size*2, self.out_size), nn.LogSoftmax(dim=0))
            else:
                self.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(feat_size, self.out_size), nn.LogSoftmax(dim=0))

    def forward(self, inputs):
        skip = [[]]*len(self.features)
        x = inputs
        for i, l in enumerate(self.features):
            x = l(x, skip[i])
            for to_block in self.connections[i]:
                skip[to_block] = skip[to_block]+[x]
        if (self.task == "classification" or self.task == "regression"):
            x = torch.flatten(x, start_dim=1)
            for l in self.fc:
                x = l(x)
        else:
            x = nn.functional.interpolate(x, size = [inputs.shape[2], inputs.shape[3]])
        del skip
        return x

    def mutate(self, mut_amount = 1, probs = {"add_b" : 0.4, "delete_b" : 0.4, "modify_b" : 0.2}, verbose = False):
        operations = list(probs.keys())
        
        backup_features = copy.deepcopy(self.features)
        backup_fc = copy.deepcopy(self.fc)
        backup_connections = copy.deepcopy(self.connections)
        
        valid_model = False
        while not valid_model:
            try:
                mutations = random.choices(operations, weights = [probs[operation] for operation in operations], k = mut_amount)
                for mut in mutations:
                    match mut:
                        case "add_b":
                            self.add_block(verbose)
                        case "delete_b":
                            if len(self.features) > 1: # If we can't delete any block, we add one instead
                                self.delete_block(verbose)
                            else:
                                self.add_block(verbose)
                        case "modify_b":
                            self.modify_block(verbose)
    
                for i in range(len(self.connections)):
                    self.connections[i].sort() # We order the connections so that they are tidy
                
                self.build_fc()
                valid_model = True
                
            except Exception as e:
                if verbose:
                    print(e)
                self.features = copy.deepcopy(backup_features)
                self.fc = copy.deepcopy(backup_fc)
                self.connections = copy.deepcopy(backup_connections)
    
    def add_block(self, verbose = False):
        # We chose a random block to add another one after
        index = random.randint(0, len(self.features)-1)
        if verbose:
            print("Added block between blocks", index, "and " + str(index+1)+".")

        self.connections = self.connections[:index+1] + [[]] + self.connections[index+1:] # We add space for the new block to make connections
        self.connections = [[b[c]*(b[c] <= index) + (b[c]+1)*((b[c]) > index) for c in range(len(b))] for b in self.connections] # And we adjust the indexes of all connections

        # We chose the hyperparameters for the new block
        if self.task not in ["classification", "regression"]:
            resizing_type = "" # If we need the output size to remain constant we do not add a resizing
        else:
            resizing_type = random.choice(["U", "D", ""])
        kernel_size = random.randint(0,int((self.max_k-1)/2))*2+1
        
        features = nn.Sequential(*list(self.features.children())[:index+1], # We keep the previous blocks, add a block with in and out channels equal to the output channels of the previous block
                                  block(in_c = self.features[index].out_c, out_c = self.features[index].out_c, resizing_type = resizing_type, from_block_channels = 0, kernel_size = kernel_size),
                                 *list(self.features.children())[index+1:]) # And then add the remaining blocks
        
        self.features = features

    def delete_block(self, verbose = False):
        # We chose a random block to delete
        index = random.randint(0, len(self.features)-1)
        if verbose:
            print("Block", index, "deleted.")

        self.connections.pop(index) # We delete the skip connections of the deleted block
        self.connections = [[b[c]*(b[c] < index) + (b[c]-1)*((b[c]) > index) for c in range(len(b)) if index != b[c]] for b in self.connections] # We adjust the indexes of all connections
        self.connections = [[b[c] for c in range(len(b)) if b[c]-1 != i] for i, b in enumerate(self.connections)] # And we remove the skip connections btween contiguous blocks

        if index == len(self.features)-1: # If it's the last block, we just remove it
            self.features = nn.Sequential(*list(self.features.children())[:index])
        else: # If not, we have to modify the input channels of the subsequent block and add the rest of the blocks onwards, taking the new skipping connections into consideration
            from_blocks = [i for i, x in enumerate(self.connections) if index in x]
            from_block_channels = 0
            for from_block in from_blocks:
                from_block_channels += self.features[from_block].out_c
                    
            features = nn.Sequential(*list(self.features.children())[:index],
                                          block(in_c = self.features[index].in_c, out_c = self.features[index+1].out_c, resizing_type = self.features[index+1].resizing_type, from_block_channels = from_block_channels, kernel_size = self.features[index+1].kernel_size))
            
            for l, blk in enumerate(list(self.features.children())[index+2:]):
                from_blocks = [i for i, x in enumerate(self.connections) if index+1+l in x]
                from_block_channels = 0
                for from_block in from_blocks:
                    from_block_channels += features[from_block].out_c

                features = nn.Sequential(*list(features.children()), block(in_c = blk.in_c, out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = from_block_channels, kernel_size = blk.kernel_size))

            self.features = features

    def modify_block(self, verbose = False):
        # We chose a random block to modify
        index = random.randint(0, len(self.features)-1)
        if verbose:
            print("Block", index, "modified.")

        out_c = 2**random.randint(self.logmin_c, self.logmax_c)
        resizing_type = random.choice(["U", "D", ""])
        kernel_size = random.randint(0,int((self.max_k-1)/2))*2+1
        features = nn.Sequential(*list(self.features.children())[:index], # We keep the previous blocks and add a modified block
                                  block(in_c = self.features[index].in_c, out_c = out_c, resizing_type = resizing_type, from_block_channels = self.features[index].from_block_channels, kernel_size = kernel_size))
        
        for l, blk in enumerate(list(self.features.children())[index+1:]): # We add the next blocks, modifying the connections accordingly and adapting the input channels on the first one
            from_blocks = [i for i, x in enumerate(self.connections) if index+1+l in x]
            from_block_channels = 0
            for from_block in from_blocks:
                from_block_channels += features[from_block].out_c

            features = nn.Sequential(*list(features.children()),
                                     block(in_c = out_c, out_c = blk.out_c, resizing_type = blk.resizing_type, from_block_channels = from_block_channels, kernel_size = blk.kernel_size))

            out_c = blk.out_c
        
        self.features = features

def weight_reset(layer):
    if hasattr(layer, 'reset_parameters'):
        layer.reset_parameters()

### 0.5. Dataset retrieval with Data Augmentation

In [6]:
def retrieve_datasets(transforms, val_len = 5000, val_da = 10):
    trainset = datasets.CIFAR10(root='./data', train=True, download=[False if os.path.isfile(".\\data\\cifar-10-python.tar.gz") else True][0], transform=transforms)
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)
    # To compare the models fairly, we take val_da augmented copies of each validation image to create a fixed but augmented validation partition
    val_dset = []
    for _ in range(val_da):
        val_dset += [trainset[i] for i in range(val_len)]
    val_loader = torch.utils.data.DataLoader(val_dset, batch_size=32, shuffle=False)
    train_loader.dataset.data = train_loader.dataset.data[val_len:,:,:]
    train_loader.dataset.targets = train_loader.dataset.targets[val_len:]
    return val_loader, train_loader


# 1. Random model generation for CIFAR-10

In this section, models will be generated at random and trained with 3 Data Augmentation schemes besides to compare with a non-augmented training. The first one consists on a series of geometrical operations, the second one consists mainly on a gaussian filter with a very big filter (which is meant to degrade the images) and the third one utilizes PyTorch's AutoAugment policy for CIFAR-10.

### 1.1. Define the transforms with and without data augmentation

In [7]:
transforms_vanilla = v2.Compose([v2.ToTensor(),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug1 = v2.Compose([v2.ToTensor(),
                                 v2.RandomHorizontalFlip(p=0.5),
                                 v2.RandomAffine(degrees=30, scale=(0.9, 1.1), shear=10),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug2 = v2.Compose([v2.ToTensor(),
                                 v2.GaussianBlur(kernel_size=9, sigma=(0.5,1.5)),
                                 v2.RandomHorizontalFlip(p=0.5),
                                 v2.RandomEqualize(p = 0.5),
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms_dataug3 = v2.Compose([v2.ToTensor(),
                                 v2.AutoAugment(v2.AutoAugmentPolicy.CIFAR10), # We use Pytorch's AutoAugment for CIFAR10
                                 v2.ToDtype(torch.float32, scale=True),
                                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                                ])

transforms = [transforms_vanilla, transforms_dataug1, transforms_dataug2, transforms_dataug3, transforms_vanilla] # The second transforms_vanilla is for the reinitialized weights case

# Download and load the test data, with no augmentations
testset = datasets.CIFAR10(root='./data', train=False, download=[False if os.path.isfile(".\\data\\cifar-10-python.tar.gz") else True][0], transform=transforms_vanilla)
test_loader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

criterion = nn.NLLLoss()

Files already downloaded and verified


### 1.2. Generate original model

In [8]:
# Model and training parameters
n_tests = 16
max_epochs = 256
patience = 4

model_args = {"dataloader" : test_loader, # To check the input and output sizes, type of problem, etc, from
              "task" : "infere",
              "out_size" : None,
              "logmin_c" : 4, # Number of channels per layer goes from 16 to 1024
              "logmax_c" : 10,
              "max_k" : 7, # Kernel size is at most 7 px wide
              "max_iter_time" : 2, # Only train models that require at most 2 seconds per batch
              "verbose" : True}

In [9]:
# Create folders to save models at
os.makedirs("saved_models/", exist_ok = True)

model_save_path = "saved_models/cifar10_mutation_original/"
os.makedirs(model_save_path, exist_ok = True)
for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)
    
seed_value = 0
valid_original_model = False
while not valid_original_model:
    set_seed(seed_value)
    seed_value += 1
    try:
        original_model = optimized_network(**model_args, n_blocks = 8, complex_classifier = random.randint(0, 1))
        valid_original_model, _ = mut_trainloop(original_model, mutations = 0, seed = seed_value, transforms = transforms, max_epochs = max_epochs, lr = 1e-3, patience = patience, model_save_path = model_save_path)
    except:
        print("Generated useless model, retrying")
    
print("Original model:", original_model)

Data Augmentation scheme:   0%|          | 0/5 [00:00<?, ?it/s]

Files already downloaded and verified


Training model:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch progress:   0%|          | 0/1407 [00:00<?, ?it/s]

CUDA out of memory. Tried to allocate 2.35 GiB. GPU 0 has a total capacity of 23.65 GiB of which 1.80 GiB is free. Process 161061 has 1.38 GiB memory in use. Process 171517 has 1.89 GiB memory in use. Process 1483150 has 388.00 MiB memory in use. Including non-PyTorch memory, this process has 18.18 GiB memory in use. Of the allocated memory 16.20 GiB is allocated by PyTorch, and 1.47 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Generated useless model, retrying


...

Calculating validation loss:   0%|          | 0/1563 [00:00<?, ?it/s]

Calculating accuracy:   0%|          | 0/1407 [00:00<?, ?it/s]

Calculating accuracy:   0%|          | 0/1563 [00:00<?, ?it/s]

Calculating accuracy:   0%|          | 0/313 [00:00<?, ?it/s]

Achieved accuracies: {'train': [0.9165111940298507, 0.8478144989339019, 0.8374200426439232, 0.8678704690831557, 0.8853500355366027], 'val': [0.7949256238003839, 0.794525751759437, 0.7439419385796545, 0.7687739923224568, 0.7977047344849648], 'test': [0.786841054313099, 0.8238817891373802, 0.7845447284345048, 0.8225838658146964, 0.7778554313099042]}
Training time: [1112.1779091358185, 3015.4224519729614, 3211.763417005539, 3535.628020763397, 952.5548124313354]
Original model: optimized_network(
  (features): Sequential(
    (0): block(
      (resizing): ConvTranspose2d(3, 3, kernel_size=(2, 2), stride=(2, 2))
      (conv): conv_block(
        (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(1, 1), padding=(4, 4))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU()
      )
    )
    (1): block(
      (resizing): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
      (conv): conv_block(
  

### 1.3 Create similar models via single mutation operation

In [15]:
base_model = "saved_models/cifar10_mutation_original/"
model_save_path = "saved_models/cifar10_1mutation/"
os.makedirs(model_save_path, exist_ok = True)
for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)

tr_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
val_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
test_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
train_times = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}

seed_value = 0
with tqdm(total=n_tests, leave = False, desc='Generating and training models') as pbar_tests:
    while len(val_accs["vanilla"]) < n_tests:
        seed_value += 1
        try:
            free_gpu_cache()
            # Now we copy the original model and mutate it just once before training
            accs, times = mut_trainloop(base_model, mutations = 1, seed = seed_value, transforms = transforms, max_epochs = max_epochs, lr = 1e-3, patience = patience, model_save_path = model_save_path)
            if not accs == None:
                tr_accs["vanilla"] = tr_accs["vanilla"] + [accs["train"][0]]
                tr_accs["daug1"] = tr_accs["daug1"] + [accs["train"][1]]
                tr_accs["daug2"] = tr_accs["daug2"] + [accs["train"][2]]
                tr_accs["daug3"] = tr_accs["daug3"] + [accs["train"][3]]
                tr_accs["reinit"] = tr_accs["reinit"] + [accs["train"][4]]
                
                val_accs["vanilla"] = val_accs["vanilla"] + [accs["val"][0]]
                val_accs["daug1"] = val_accs["daug1"] + [accs["val"][1]]
                val_accs["daug2"] = val_accs["daug2"] + [accs["val"][2]]
                val_accs["daug3"] = val_accs["daug3"] + [accs["val"][3]]
                val_accs["reinit"] = val_accs["reinit"] + [accs["val"][4]]
                
                test_accs["vanilla"] = test_accs["vanilla"] + [accs["test"][0]]
                test_accs["daug1"] = test_accs["daug1"] + [accs["test"][1]]
                test_accs["daug2"] = test_accs["daug2"] + [accs["test"][2]]
                test_accs["daug3"] = test_accs["daug3"] + [accs["test"][3]]
                test_accs["reinit"] = test_accs["reinit"] + [accs["test"][4]]
    
                train_times["vanilla"] = train_times["vanilla"] + [times[0]]
                train_times["daug1"] = train_times["daug1"] + [times[1]]
                train_times["daug2"] = train_times["daug2"] + [times[2]]
                train_times["daug3"] = train_times["daug3"] + [times[3]]
                train_times["reinit"] = train_times["reinit"] + [times[4]]

                Path("tr_accs_1mut.json").write_text(json.dumps(tr_accs))
                Path("val_accs_1mut.json").write_text(json.dumps(val_accs))
                Path("test_accs_1mut.json").write_text(json.dumps(test_accs))
                Path("train_times_1mut.json").write_text(json.dumps(train_times))

                pbar_tests.update()
            else:
                print("Generated useless model, retrying")
                
        except Exception as e:
            print(e)

Generating and training models:   0%|          | 0/16 [00:00<?, ?it/s]

Exception ignored in: <function tqdm.__del__ at 0x73a77efe8e50>
Traceback (most recent call last):
  File "/home/adgomezm/.local/lib/python3.10/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/adgomezm/.local/lib/python3.10/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Data Augmentation scheme:   0%|          | 0/5 [00:00<?, ?it/s]

Added block between blocks 1 and 2.
Files already downloaded and verified


Training model:   0%|          | 0/256 [00:00<?, ?it/s]

...

Calculating validation loss:   0%|          | 0/1563 [00:00<?, ?it/s]

Calculating accuracy:   0%|          | 0/1407 [00:00<?, ?it/s]

Calculating accuracy:   0%|          | 0/1563 [00:00<?, ?it/s]

Calculating accuracy:   0%|          | 0/313 [00:00<?, ?it/s]

Achieved accuracies: {'train': [0.9019411869225302, 0.8381974058280028, 0.837930881307747, 0.8880152807391614, 0.9369003198294243], 'val': [0.8021633077415227, 0.8001439539347409, 0.7499000319897633, 0.7819097888675623, 0.7986644273832374], 'test': [0.7924321086261981, 0.8244808306709265, 0.7926317891373802, 0.8388578274760383, 0.7970247603833865]}


In [16]:
print("tr_accs_1mut =", tr_accs)
print("val_accs_1mut =", val_accs)
print("test_accs_1mut =", test_accs)
print("train_times_1mut =", train_times)

tr_accs_1mut = {'vanilla': [0.9161114072494669, 0.8742226368159204, 0.9321695095948828, 0.9163779317697228, 0.8557658137882018, 0.8539001421464109, 0.9095815565031983, 0.9448738450604123, 0.9192208599857854, 0.8804637526652452, 0.8908582089552238, 0.9263948116560057, 0.9032293887704336, 0.9306147832267235, 0.8324449182658138, 0.9019411869225302], 'daug1': [0.8661158493248046, 0.8527674129353234, 0.8904806325515281, 0.9052505330490405, 0.8545442430703625, 0.8195851101634684, 0.8761549395877755, 0.855432658137882, 0.8736451670220327, 0.8841284648187633, 0.8817963752665245, 0.8395522388059702, 0.8931458777540867, 0.8219838308457711, 0.7971526297085999, 0.8381974058280028], 'daug2': [0.8463264036958067, 0.8218949893390192, 0.8648054371002132, 0.8958777540867093, 0.8409514925373134, 0.7827825159914712, 0.8283359985785359, 0.8305570362473348, 0.8492359630419332, 0.8616515636105189, 0.8428837953091685, 0.8361540511727079, 0.8494136460554371, 0.8219616204690832, 0.7894678393745558, 0.837930881

### 1.4 Create similar models via three mutation operations

In [ ]:
base_model = "saved_models/cifar10_mutation_original/"
model_save_path = "saved_models/cifar10_3mutations/"
os.makedirs(model_save_path, exist_ok = True)
for t in range(len(transforms)):
    os.makedirs(model_save_path+"data_aug_"+str(t), exist_ok = True)


if os.path.exists('tr_accs_3mut.json'):
    with open('tr_accs_3mut.json', 'r') as file:
        tr_accs = json.load(file)
    with open('val_accs_3mut.json', 'r') as file:
        val_accs = json.load(file)
    with open('test_accs_3mut.json', 'r') as file:
        test_accs = json.load(file)
    with open('train_times_3mut.json', 'r') as file:
        train_times = json.load(file)
else:
    tr_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    val_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    test_accs = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}
    train_times = {"vanilla" : [],"daug1" : [],"daug2" : [],"daug3" : [], "reinit" : []}

with tqdm(total=n_tests, leave = False, desc='Generating and training models') as pbar_tests:
    pbar_tests.update(len(val_accs["vanilla"]))
    while len(val_accs["vanilla"]) < n_tests:
        seed_value += 1
        try:
            free_gpu_cache()
            # Now we copy the original model and mutate 3 times before training
            accs, times = mut_trainloop(base_model, mutations = 1, seed = seed_value, transforms = transforms, max_epochs = max_epochs, lr = 1e-3, patience = patience, model_save_path = model_save_path)
            
            if not accs == None:
                tr_accs["vanilla"] = tr_accs["vanilla"] + [accs["train"][0]]
                tr_accs["daug1"] = tr_accs["daug1"] + [accs["train"][1]]
                tr_accs["daug2"] = tr_accs["daug2"] + [accs["train"][2]]
                tr_accs["daug3"] = tr_accs["daug3"] + [accs["train"][3]]
                tr_accs["reinit"] = tr_accs["reinit"] + [accs["train"][4]]
                
                val_accs["vanilla"] = val_accs["vanilla"] + [accs["val"][0]]
                val_accs["daug1"] = val_accs["daug1"] + [accs["val"][1]]
                val_accs["daug2"] = val_accs["daug2"] + [accs["val"][2]]
                val_accs["daug3"] = val_accs["daug3"] + [accs["val"][3]]
                val_accs["reinit"] = val_accs["reinit"] + [accs["val"][4]]
                
                test_accs["vanilla"] = test_accs["vanilla"] + [accs["test"][0]]
                test_accs["daug1"] = test_accs["daug1"] + [accs["test"][1]]
                test_accs["daug2"] = test_accs["daug2"] + [accs["test"][2]]
                test_accs["daug3"] = test_accs["daug3"] + [accs["test"][3]]
                test_accs["reinit"] = test_accs["reinit"] + [accs["test"][4]]
    
                train_times["vanilla"] = train_times["vanilla"] + [times[0]]
                train_times["daug1"] = train_times["daug1"] + [times[1]]
                train_times["daug2"] = train_times["daug2"] + [times[2]]
                train_times["daug3"] = train_times["daug3"] + [times[3]]
                train_times["reinit"] = train_times["reinit"] + [times[4]]

                Path("tr_accs_3mut.json").write_text(json.dumps(tr_accs))
                Path("val_accs_3mut.json").write_text(json.dumps(val_accs))
                Path("test_accs_3mut.json").write_text(json.dumps(test_accs))
                Path("train_times_3mut.json").write_text(json.dumps(train_times))

                pbar_tests.update()
            else:
                print("Generated useless model, retrying")
                
        except Exception as e:
            print(e)


Generating and training models:   0%|          | 0/16 [00:00<?, ?it/s]

Data Augmentation scheme:   0%|          | 0/5 [00:00<?, ?it/s]

Block 7 deleted.
Files already downloaded and verified


Training model:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch progress:   0%|          | 0/1407 [00:00<?, ?it/s]

...

Calculating accuracy:   0%|          | 0/1563 [00:00<?, ?it/s]

Calculating accuracy:   0%|          | 0/313 [00:00<?, ?it/s]

Block 7 deleted.
Files already downloaded and verified


Training model:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch progress:   0%|          | 0/1407 [00:00<?, ?it/s]

In [ ]:
print("tr_accs_3mut =", tr_accs)
print("val_accs_3mut =", val_accs)
print("test_accs_3mut =", test_accs)
print("train_times_3mut =", train_times)